In [1]:
import pandas as pd
import numpy as np

In [2]:
## test to make sure this file is up and running
print("hello world")

hello world


## Set up connection to MySQL database

In [3]:
## set up connection to MySQL database
import mysql.connector
from mysql.connector import Error


In [4]:
config = {
    'host': '172.29.218.123',
    'user': 'tg800',
    'password': 'b4R0Gm5psOvtmkPguM5aZFsF',
    'database': 'CGE'
}

In [5]:
conn = mysql.connector.connect(**config)

# **config unpacks the dictionary above into the connection settings
# Like logging into MySQL Workbench but through Python

cursor = conn.cursor()
## acts like a "pen" that executed the SQL commands on the server

## Individual data imports

#### Stress_suppl RRI

In [64]:
stress_suppl_RRI_file = pd.read_excel('/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/Stress_Suppl_RRI_Spectra.xlsx')

file_name = 'stress_suppl_RRI_TG'
schema = 'stress_suppl'

stress_suppl_RRI_file.head()

,SubID,Peak_B1,Peak_6P
0,210,22767.65820,6.286232e+04
1,211,6286.81787,5.519914e+05
2,215,4871.19092,3.264948e+05
3,216,41480.94531,6.326689e+05
4,217,68828.31250,1.082345e+06


In [67]:
stress_suppl_rri_df = pd.melt(
    stress_suppl_RRI_file,
    id_vars = 'SubID',
    value_vars = ['Peak_B1', 'Peak_6P'],
    var_name = 'timepoint',
    value_name= 'RRI'

)
stress_suppl_rri_df['SubID'] = stress_suppl_rri_df['SubID'] + 120000
stress_suppl_rri_df.rename({'SubID': 'ID6'}, axis=1, inplace=True)

stress_suppl_rri_df.head()
## Looks good

,ID6,timepoint,RRI
0,120210,Peak_B1,22767.65820
1,120211,Peak_B1,6286.81787
2,120215,Peak_B1,4871.19092
3,120216,Peak_B1,41480.94531
4,120217,Peak_B1,68828.31250


#### BAA RRI

In [26]:
BAA_rri_csv = pd.read_excel('/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/BAA_RRI.xlsx')

BAA_rri_csv.head()

,SubID,Peak_B1,Peak_6P
0,1000,2648.09619,2.547655e+05
1,1001,105498.84375,6.894281e+05
2,1002,34559.87109,1.059542e+06
3,1003,9710.73926,1.075023e+06
4,1005,23163.79102,1.062483e+06


In [28]:
BAA_rri_csv_df = BAA_rri_csv.copy()
table_name = 'BAA_RRI_df_TG'
schema = 'BAA'

In [42]:
BAA_rri_df = pd.melt(
    BAA_rri_csv_df,
    id_vars = 'SubID',
    value_vars = ['Peak_B1', 'Peak_6P'], 
    var_name = 'timepoint',
    value_name = 'RRI'
)


BAA_rri_df.replace({'timepoint':{'Peak_B1': 'B1', 'Peak_6P':'6P'}}, inplace=True)
BAA_rri_df.rename({'SubID':'ID6'}, axis=1, inplace=True)
BAA_rri_df['ID6'] = BAA_rri_df['ID6'] + 140000

BAA_rri_df.head()
## data looks good and ready to be imported

,ID6,timepoint,RRI
0,141000,B1,2648.09619
1,141001,B1,105498.84375
2,141002,B1,34559.87109
3,141003,B1,9710.73926
4,141005,B1,23163.79102


In [43]:
baa_rri_cols = []
for col in BAA_rri_df.columns:
    sample = BAA_rri_df[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    baa_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(baa_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")


Table `BAA_RRI_df_TG` dropped if it existed
Table `BAA_RRI_df_TG` is ready! to import into BAA


In [44]:
## Insert data

inserted = 0

for _, row in BAA_rri_df.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in BAA_rri_df.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

Done! 148 rows successfully inserted.


### Stress Supplement Physio RRI Peaks

In [6]:
stress_suppl_physio_df = pd.read_excel('/Users/thomasgooding/Desktop/remissingspectrumdata/stress_supp_physio_needpeaks.xlsx')

stress_suppl_physio_df.head()

,file,TempMin,TempMax,TempMean,TempDev,SCMin,SCMax,SCMean,SCDev,SCfMin,...,UniqID,Gender,History,Session,LendARM,SCbase,Set,Study,Task,id6
0,210FN18503006AS6P,33.11,34.45,33.939,0.202,-5.61,-1.76,-4.031,1.079,-0.708,...,210,F,N,1,850,30,6,AS,6P,120210
1,210FN18503006ASB1,29.45,30.25,29.735,0.149,-1.71,11.01,1.968,2.938,-2.962,...,210,F,N,1,850,30,6,AS,B1,120210
2,210FN18503006ASNg1,32.64,34.05,33.455,0.249,-7.36,-4.34,-6.255,0.704,-0.878,...,210,F,N,1,850,30,6,AS,NG1,120210
3,210FN18503006ASNg2,32.46,33.35,33.007,0.139,-8.10,-6.77,-7.509,0.404,-0.298,...,210,F,N,1,850,30,6,AS,NG2,120210
4,210FN18503006ASNg3,31.51,32.39,32.073,0.193,-6.09,-3.09,-4.843,0.879,-0.980,...,210,F,N,1,850,30,6,AS,NG3,120210


In [7]:
stress_suppl_physio_df['ID6_tg'] = stress_suppl_physio_df['file'].str[0:3].astype(int) + 120000

stress_suppl_physio_df['task_tg'] = stress_suppl_physio_df['file'].str[15:].str.upper()

stress_suppl_physio_df[['file','id6', 'ID6_tg', 'Task', 'task_tg', 'RRIMax', 'id6']].head()

,file,id6,ID6_tg,Task,task_tg,RRIMax,id6
0,210FN18503006AS6P,120210,120210,6P,6P,906,120210
1,210FN18503006ASB1,120210,120210,B1,B1,1015,120210
2,210FN18503006ASNg1,120210,120210,NG1,NG1,1523,120210
3,210FN18503006ASNg2,120210,120210,NG2,NG2,1191,120210
4,210FN18503006ASNg3,120210,120210,NG3,NG3,1301,120210


In [10]:
stress_suppl_physio_TG = stress_suppl_physio_df.copy()

table_name = 'stress_suppl_physio_TG'
schema = 'stress_suppl'

In [11]:
stress_suppl_rri_cols = []
for col in stress_suppl_physio_TG.columns:
    sample = stress_suppl_physio_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    stress_suppl_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(stress_suppl_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")


Table `stress_suppl_physio_TG` dropped if it existed
Table `stress_suppl_physio_TG` is ready! to import into stress_suppl


In [12]:
stress_suppl_physio_TG.shape

(248, 175)

In [13]:
## Insert data

inserted = 0

for _, row in stress_suppl_physio_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in stress_suppl_physio_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

Done! 248 rows successfully inserted.


### Stress pilot physio data

In [86]:
stress_pilot_physio_df = pd.read_excel('/Users/thomasgooding/Desktop/remissingspectrumdata/stress_pilot_physio.xlsx')

stress_pilot_physio_df.head()

,file,TempMin,TempMax,TempMean,TempDev,SCMin,SCMax,SCMean,SCDev,SCfMin,...,EXSPER,RSOC,RDIS,RSUP,groupN,F54,Liking,TaskDrug,Arousa,AD
0,031FNP1B1,32.82,34.87,33.809,0.709,5.981,15.696,8.302,1.703,-5.546,...,10,4,1,2,2.0,NaN,NaN,NaN,NaN,NaN
1,031FNP1A1,26.98,31.34,29.286,1.837,-2.168,9.530,2.515,2.379,-3.383,...,10,4,1,2,2.0,NaN,NaN,NaN,NaN,NaN
2,031FNP1B2,29.49,34.75,32.643,2.003,-2.405,6.881,0.844,2.049,-7.049,...,10,4,1,2,2.0,NaN,NaN,NaN,NaN,NaN
3,031FNP1Ng,29.98,30.86,30.396,0.218,-6.399,12.116,-1.894,3.174,-3.995,...,10,4,1,2,2.0,NaN,1.666667,NaN,3.733333,NaN
4,031FNP1Nt,30.01,31.38,30.356,0.349,-2.559,8.502,-0.112,1.934,-2.668,...,10,4,1,2,2.0,NaN,4.933333,NaN,2.000000,NaN


In [98]:
stress_pilot_physio_df['task_tg'] = stress_pilot_physio_df['file'].str[-2:]

stress_pilot_physio_df['ID6'] = stress_pilot_physio_df['file'].str[0:3].astype('int') + 100000

stress_pilot_physio_df[['file','ID','ID6', 'RRI', 'Task1', 'task_tg']].head()

## looks good

,file,ID,ID6,RRI,Task1,task_tg
0,031FNP1B1,31,100031,4144.100,B1,B1
1,031FNP1A1,31,100031,4807.375,A1,A1
2,031FNP1B2,31,100031,902.800,B2,B2
3,031FNP1Ng,31,100031,30050.700,Ng,Ng
4,031FNP1Nt,31,100031,14606.200,Nt,Nt


In [104]:
stress_pilot_physio_df.shape

(324, 232)

In [100]:
stress_pilot_physio_TG = stress_pilot_physio_df.copy()

table_name = 'stress_pilot_physio_TG'
schema = 'stress_pilot'

In [101]:
stress_pilot_rri_cols = []
for col in stress_pilot_physio_TG.columns:
    sample = stress_pilot_physio_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    stress_pilot_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(stress_pilot_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")


Table `stress_pilot_physio_TG` dropped if it existed
Table `stress_pilot_physio_TG` is ready! to import into stress_pilot


In [103]:
len(stress_pilot_physio_TG)

324

In [105]:
## Insert data

inserted = 0

for _, row in stress_pilot_physio_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in stress_pilot_physio_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

Done! 324 rows successfully inserted.


In [73]:
stress_pic_physio_tg = pd.read_excel('/Users/thomasgooding/Desktop/remissingspectrumdata/stress_pictword_physio.xlsx')

stress_pic_physio_tg.head()

,file,TempMin,TempMax,TempMean,TempDev,SCMin,SCMax,SCMean,SCDev,SCfMin,...,SC,RRI,PULSE,PTT,TF,Phase,Coher,phase2,Phase3,id6
0,001FFA1111S6P.,34.40,34.74,34.595,0.099,-19.86,-18.08,-19.242,0.416,-8.938,...,0.00100,871433.0,425.3,3906.1,0.310,-167.5,0.993,192.5,177.5,110001
1,001FFA1111SAL.,35.89,36.04,35.975,0.047,-12.57,-10.67,-11.780,0.527,-0.056,...,0.00100,6910.5,24.3,37.2,0.220,-124.9,0.560,235.1,220.1,110001
2,001FFA1111SB1.,33.47,34.85,34.357,0.431,-17.56,-1.87,-10.275,4.351,-7.744,...,0.00003,1041.5,70.8,27.1,0.260,-150.8,0.346,209.2,194.2,110001
3,001FFA1111SB2.,35.11,35.80,35.647,0.175,-5.04,20.66,3.305,5.548,-4.433,...,0.10000,1309.6,20.2,1.8,0.353,-133.4,0.366,226.6,211.6,110001
4,001FFA1111SNg.,35.71,35.90,35.826,0.047,-9.01,1.14,-5.780,2.348,-5.029,...,0.13000,6825.6,4.7,18.4,0.100,-142.9,0.291,217.1,202.1,110001


In [77]:
stress_pic_physio_tg['file'] = stress_pic_physio_tg['file'].str.replace('.', '')

stress_pic_physio_tg['task_tg'] = stress_pic_physio_tg['file'].str[-2:].str.upper()

stress_pic_physio_tg[['file', 'id6', 'task_tg', 'Task', 'RRIMax']].head()

,file,id6,task_tg,Task,RRIMax
0,001FFA1111S6P,110001,6P,6P,966
1,001FFA1111SAL,110001,AL,AL,752
2,001FFA1111SB1,110001,B1,B1,859
3,001FFA1111SB2,110001,B2,B2,746
4,001FFA1111SNg,110001,NG,NG,858


In [82]:
file_name = 'stress_pic_physio_tg'
table_name = 'stress_pic_physio_tg'
schema = 'stress_pic'

In [83]:
stress_pic_rri_cols = []
for col in stress_pic_physio_tg.columns:
    sample = stress_pic_physio_tg[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    stress_pic_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(stress_pic_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")


Table `stress_pic_physio_tg` dropped if it existed
Table `stress_pic_physio_tg` is ready! to import into stress_pic


In [84]:
len(stress_pic_physio_tg)

2796

In [85]:
## Insert data

inserted = 0

for _, row in stress_pic_physio_tg.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in stress_pic_physio_tg.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

Done! 2796 rows successfully inserted.


### stress_prevention_physio

In [63]:
stress_prevention_physio_df = pd.read_excel('/Users/thomasgooding/Desktop/remissingspectrumdata/prev_physio.xlsx')

stress_prevention_physio_df.head()


,file,TempMin,TempMax,TempMean,TempDev,SCMin,SCMax,SCMean,SCDev,SCfMin,...,Task,Ord,Order,RV,SC,RRI,PULSE,PTT,RVTMean,RVAMean
0,501FNC1122P6P.,30.35,31.79,31.026,0.441,0.84,9.02,3.052,2.326,-0.522,...,6P,.,6PBefore,7204328.0,0.00200,1191249.0,323.8,11573.5,830.324,340.735
1,501FNC1122PAL.,30.41,32.16,31.410,0.527,0.48,0.92,0.706,0.115,-0.022,...,AL,.,6PBefore,49116.7,0.00010,61283.9,14.7,346.1,181.665,83.662
2,501FNC1122PB1.,30.63,32.01,31.392,0.422,0.22,7.40,1.662,1.803,-1.514,...,B1,.,6PBefore,682.9,0.00000,24043.1,277.9,95.1,197.844,121.756
3,501FNC1122PEc.,31.92,32.18,32.059,0.070,0.88,1.02,0.942,0.034,-0.021,...,EC,.,6PBefore,124362.1,0.00013,99240.1,31.3,344.5,186.151,122.147
4,501FNC1122PMa.,31.47,32.16,31.850,0.229,1.03,1.39,1.191,0.097,-0.014,...,MA,.,6PBefore,81396.7,0.00004,41287.1,76.4,304.2,169.037,590.555


In [66]:
# stress_prevention_physio_df['ID6_tg'] = stress_prevention_physio_df['file'].str[0:3].astype(int) + 130000

stress_prevention_physio_df['file'] = stress_prevention_physio_df['file'].str.replace('.', '')
stress_prevention_physio_df['task_tg'] = stress_prevention_physio_df['file'].str[-2:].str.upper()

stress_prevention_physio_df[['file', 'id6','Task', 'task_tg', 'Session', 'RRI']].head()

,file,id6,Task,task_tg,Session,RRI
0,501FNC1122P6P,130501,6P,6P,1,1191249.0
1,501FNC1122PAL,130501,AL,AL,1,61283.9
2,501FNC1122PB1,130501,B1,B1,1,24043.1
3,501FNC1122PEc,130501,EC,EC,1,99240.1
4,501FNC1122PMa,130501,MA,MA,1,41287.1


In [69]:
stress_prevention_physio_TG = stress_prevention_physio_df.copy()
table_name = 'stress_prevention_physio_TG'
schema = 'stress_prevention'

In [68]:
# stress_prevention_physio_df.columns = stress_prevention_physio_df.columns.str.upper()

# stress_prevention_physio_df

In [70]:
stress_prev_rri_cols = []
for col in stress_prevention_physio_TG.columns:
    sample = stress_prevention_physio_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    stress_prev_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(stress_prev_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")


Table `stress_prevention_physio_TG` dropped if it existed
Table `stress_prevention_physio_TG` is ready! to import into stress_prevention


In [71]:
len(stress_prevention_physio_TG)

1035

In [72]:
## Insert data

inserted = 0

for _, row in stress_prevention_physio_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in stress_prevention_physio_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

Done! 1035 rows successfully inserted.


### BAA_physio

In [10]:
baa_physio_needpeaks = pd.read_excel('/Users/thomasgooding/Desktop/remissingspectrumdata/baa_physio_needpeaks.xlsx')

baa_physio_needpeaks.head()

,file,TempMin,TempMax,TempMean,TempDev,SCMin,SCMax,SCMean,SCDev,SCfMin,...,Gender,Session,History,Group,Age,Block,LengARM,SCbase,Task,PWV
0,1000F1SA211684018AL,33.69,34.97,33.971,0.356,16.89,27.08,19.530,2.103,-2.117,...,F,1,S,A,21,16,840,18,AL,2.835606
1,1000F1SA211684018B1,34.33,35.16,34.784,0.325,8.38,13.20,9.094,0.926,-0.761,...,F,1,S,A,21,16,840,18,B1,2.717787
2,1000F1SA211684018B2,34.54,34.73,34.638,0.056,14.09,16.41,15.047,0.585,-1.588,...,F,1,S,A,21,16,840,18,B2,2.722491
3,1000F1SA211684018Ng,33.08,33.69,33.428,0.192,18.70,27.42,20.663,1.483,-2.265,...,F,1,S,A,21,16,840,18,NG,2.874822
4,1000F1SA211684018Nt,32.59,33.11,32.947,0.111,16.03,22.47,18.066,1.502,-0.647,...,F,1,S,A,21,16,840,18,NT,2.880747


In [13]:
## Isolate subject ID and task subtype for later mapping to MySQL database

baa_physio_needpeaks['task_tg'] = baa_physio_needpeaks['file'].str[-2:].str.upper()

baa_physio_needpeaks['ID6'] = (baa_physio_needpeaks['file'].str[0:4]).astype(int)
baa_physio_needpeaks['ID6'] = baa_physio_needpeaks['ID6'] + 140000

baa_physio_needpeaks[['file', 'task_tg', 'Task', 'ID6']].head()

## I duplicated the task  column but will leave that for now in case there's a discrepancy.

,file,task_tg,Task,ID6
0,1000F1SA211684018AL,AL,AL,141000
1,1000F1SA211684018B1,B1,B1,141000
2,1000F1SA211684018B2,B2,B2,141000
3,1000F1SA211684018Ng,NG,NG,141000
4,1000F1SA211684018Nt,NT,NT,141000


In [14]:
BAA_physio_peaks_TG = baa_physio_needpeaks.copy()
table_name = 'BAA_physio_peaks_TG'
schema = 'BAA'

In [15]:
baa_rri_cols = []
for col in BAA_physio_peaks_TG.columns:
    sample = BAA_physio_peaks_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    baa_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(baa_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")

Table `BAA_physio_peaks_TG` dropped if it existed
Table `BAA_physio_peaks_TG` is ready! to import into BAA


In [23]:
# list(BAA_physio_peaks_TG.columns.sort_values())

In [ ]:
len(BAA_physio_peaks_TG)

575

In [16]:
## Insert data

inserted = 0

for _, row in BAA_physio_peaks_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in BAA_physio_peaks_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

Done! 575 rows successfully inserted.


### CGE RRI and RVFMean

In [ ]:
CGE_RRI_RVFMean_csv = '/Users/thomasgooding/Desktop/CGE_RRI_RVFMean_TG.csv'
table_name = 'CGE_RRI_RVFMean_TG'
schema = 'CGE'

In [ ]:
CGE_RRI_RVFMean_TG = pd.read_csv(CGE_RRI_RVFMean_csv, encoding= 'utf-8-sig')

CGE_RRI_RVFMean_TG.rename(columns={'MAX_B1':'B1_RRI', 'F_B1': 'B1_RVFMean', 'F_6P': 'P6_RVFMean', 'MAX_6P': 'P6_RRI'}, inplace=True)
CGE_RRI_RVFMean_TG.head()

In [ ]:
cge_rri_cols = []
for col in CGE_RRI_RVFMean_TG.columns:
    sample = CGE_RRI_RVFMean_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    cge_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(cge_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")

In [ ]:
## Insert data

inserted = 0

for _, row in CGE_RRI_RVFMean_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in CGE_RRI_RVFMean_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

### VT2 RRI and RVFMean import

In [14]:
VT2_RRI_RVFMean_csv = '/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/VT2_RRI_RVFMean_TG.csv'

table_name = 'VT2_RRI_RVFMean_TG'

schema = 'VT2'

In [15]:
VT2_RRI_RVFMean_TG = pd.read_csv(VT2_RRI_RVFMean_csv, encoding= 'utf-8-sig')

VT2_RRI_RVFMean_TG.rename(columns={'MAX_B1':'B1_RRI', 'F_B1': 'B1_RVFMean', 'F_6P': 'P6_RVFMean', 'MAX_6P': 'P6_RRI'}, inplace=True)
VT2_RRI_RVFMean_TG.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/VT2_RRI_RVFMean_TG.csv'

In [ ]:
vt2_rri_cols = []
for col in VT2_RRI_RVFMean_TG.columns:
    sample = VT2_RRI_RVFMean_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    vt2_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(vt2_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")

In [ ]:
## Insert data

inserted = 0

for _, row in VT2_RRI_RVFMean_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else str(v) for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in VT2_RRI_RVFMean_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# Save changes and close connection
# ==============================
conn.commit()  ## .commit() is like hitting "save" 

print(f"Done! {inserted} rows successfully inserted.")

### Athlete Study RRI + Demo

In [ ]:
# athlete_RRI_csv = '/Users/thomasgooding/Desktop/Athlete_HRV_demo_data.csv' ## original file path
athlete_RRI_csv = '/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/completed/Athlete_HRV_demo_data.csv'

table_name = 'Athlete_RRI_demo_TG'

# schema = 'WTP' ## originally imported to WTP before I made the Athlete schema --- IGNORE ---
schema = 'Athlete'

In [ ]:
athlete_RRI_demo_TG = pd.read_csv(athlete_RRI_csv,  encoding='utf-8-sig')

athlete_RRI_demo_TG.head()


In [ ]:
athlete_rri_cols = []
for col in athlete_RRI_demo_TG.columns:
    sample = athlete_RRI_demo_TG[col].dropna()
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"

    athlete_rri_cols.append(f"`{col}` {col_type}")  # append to list

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(athlete_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready!")

In [ ]:
## Insert data

inserted = 0

for _, row in athlete_RRI_demo_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else v for v in row]
    ## returns empty/blank cells into Nulls for MySQL but keeps numeric values as numeric (not converting to strings)
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in athlete_RRI_demo_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

## Save and close connection
conn.commit()  
## .commit() is like hitting "save" - without this your inserts won't actually save

print(f"Done! {inserted} rows successfully inserted.")

### WTP import

In [ ]:
# WTP_RRI_csv_file = '/Users/thomasgooding/Desktop/WTP_RRI_TG.csv' ## Original file path
WTP_RRI_csv_file = '/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/WTP_RRI_TG.csv'

table_name = 'WTP_RRI_TG'
schema = 'WTP'

In [ ]:
WTP_RRI_TG = pd.read_csv(WTP_RRI_csv_file, encoding='utf-8-sig')

WTP_RRI_TG.drop(columns=['Unnamed: 7'], inplace=True)
WTP_RRI_TG.head()

### WTP RRI data

In [ ]:
### Create a table in MySQL based on the columns in the dataframe and their data types
wtp_rri_cols = []
for col in WTP_RRI_TG.columns:
    sample = WTP_RRI_TG[col].dropna()  # looks at actual data in each column, ignoring blanks
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"          # long text, no row size limit issues

    wtp_rri_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(wtp_rri_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready!")

In [ ]:
## Insert data

inserted = 0

for _, row in WTP_RRI_TG.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else v for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in WTP_RRI_TG.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# STEP 8: Save changes and close connection
# ==============================
conn.commit()  
## .commit() is like hitting "save" - without this your inserts won't actually save

print(f"Done! {inserted} rows successfully inserted.")

#### Run this query in MySQL to insert RRI data into existing WTP.HRV table

-- SELECT * FROM WTP.HRV;

-- SELECT * FROM WTP.WTP_RRI_TG;

SET SQL_SAFE_UPDATES = 0;

UPDATE WTP.HRV hrv
JOIN WTP.WTP_RRI_TG tg
  ON tg.UniqID = hrv.sub_id
  AND tg.Task = hrv.sub_task
  AND tg.Session = hrv.session
SET hrv.RRI = tg.RRI;

SET SQL_SAFE_UPDATES = 1;


##$ run the same for Vt1 RRI to get into VT1.HRV

-- SELECT * FROM WTP.HRV;

-- SELECT * FROM WTP.WTP_RRI_TG;

SET SQL_SAFE_UPDATES = 0;

UPDATE WTP.HRV hrv
JOIN WTP.WTP_RRI_TG tg
  ON tg.UniqID = hrv.sub_id
  AND tg.Task = hrv.sub_task
  AND tg.Session = hrv.session
SET hrv.RRI = tg.RRI;

SET SQL_SAFE_UPDATES = 1;

### WTP demographics

In [ ]:
## lets upload a csv file to test how well this works
## WTP daemographic data
# WTP_csv_file = '/Users/thomasgooding/Desktop/SQL data to import/WTP_dem_import.csv' ## Original file path
WTP_csv_file = '/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/completed/WTP_dem_import.csv'
table_name = 'WTP_dem_TG'
schema = 'WTP'

In [ ]:
## need to strip the BOM character which was preventig me from using import wizard in MySQL workbench. 
## I can modify the original csv file to a format without the BOM character, but the import wizard was slow...testing this method to see if it's ultimately faster for the several csv files I need to import.

wtp_df = pd.read_csv(WTP_csv_file, encoding='utf-8-sig')  

wtp_df.columns = wtp_df.columns.str.strip()

## get confirmation that the csv file was loaded correctly and that the rows/columns are as expected.
print(f"CSV loaded successfullly! Found {len(wtp_df)} rows and {len(wtp_df.columns)} columns")
# print(f"Columns: {list(wtp_df.columns)}")

In [ ]:
### Create a table in MySQL based on the columns in the dataframe and their data types
wtp_cols = []
for col in wtp_df.columns:
    sample = wtp_df[col].dropna()  # looks at actual data in each column, ignoring blanks
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"          # long text, no row size limit issues

    wtp_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(wtp_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready!")

In [ ]:
## Insert data

inserted = 0

for _, row in wtp_df.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else v for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in wtp_df.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1

    # ==============================
# STEP 8: Save changes and close connection
# ==============================
conn.commit()  
## .commit() is like hitting "save" - without this your inserts won't actually save

print(f"Done! {inserted} rows successfully inserted.")

### Stress_pic data import

In [ ]:
# stress_pic_dem_df = pd.read_csv(
#     '/Users/thomasgooding/Desktop/SQL data to import/stress_pic_dem_tg.csv',
#     encoding='latin1') ## original file path

stress_pic_dem_df = pd.read_csv('/Users/thomasgooding/Library/CloudStorage/OneDrive-WashingtonStateUniversity(email.wsu.edu)/TMG- Personal/Rutgers University/SQL data to import/completed/stress_pic_dem_tg.csv', encoding='latin1')

stress_pic_dem_df.columns = stress_pic_dem_df.columns.str.strip()
table_name = 'stress_pic_dem_TG'
schema = 'stress_pic'

print(f"CSV loaded: {len(stress_pic_dem_df)} rows, {len(stress_pic_dem_df.columns)} columns")

In [ ]:
stress_pic_dem_df.head()

In [ ]:
stress_pic_cols= []

for col in stress_pic_dem_df.columns:
    sample = stress_pic_dem_df[col].dropna()  # looks at actual data in each column, ignoring blanks
    
    try:
        sample.astype(float)
        # check if all values are whole numbers before assigning INT
        if (sample.astype(float) % 1 == 0).all():
            col_type = "INT"
        else:
            col_type = "FLOAT"  # has decimals, assign FLOAT
    except (ValueError, TypeError):
        max_len = sample.astype(str).str.len().max()
        if max_len <= 50:
            col_type = "VARCHAR(50)"
        elif max_len <= 255:
            col_type = "VARCHAR(255)"
        else:
            col_type = "TEXT"    
            
    stress_pic_cols.append(f"`{col}` {col_type}")

drop_statement = f"DROP TABLE IF EXISTS `{schema}`.`{table_name}`;"
cursor.execute(drop_statement)
print(f"Table `{table_name}` dropped if it existed")

create_statement = f"CREATE TABLE IF NOT EXISTS `{schema}`.`{table_name}` ({', '.join(stress_pic_cols)});"
cursor.execute(create_statement)
print(f"Table `{table_name}` is ready! to import into {schema}")

In [ ]:
inserted = 0
for _, row in stress_pic_dem_df.iterrows():
    values        = [None if pd.isna(v) else str(v) for v in row]
    placeholders  = ', '.join(['%s'] * len(values))
    stress_pic_col_names = ', '.join([f'`{c}`' for c in stress_pic_dem_df.columns])

    insert_statement = f"INSERT INTO `{stress_pic_schema}`.`{stress_pic_table_name}` ({stress_pic_col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)
    inserted += 1

conn.commit()
print(f"Done! {inserted} rows inserted into `{stress_pic_schema}`.`{stress_pic_table_name}`")

In [ ]:
## Insert data

inserted = 0

for _, row in stress_pic_dem_df.iterrows():
    # df.itterows() loops through every row in the csv file one at a time
    values = [None if pd.isna(v) else v for v in row]
    ## converts each value to a string and returns empty/blank cells into Nulls for MySQL
    placeholders = ','.join(['%s'] * len(values))

    col_names = ', '.join([f'`{c}`' for c in stress_pic_dem_df.columns])
    # builds the column name list for the INSERT statement
    
    insert_statement = f"INSERT INTO `{schema}`.`{table_name}` ({col_names}) VALUES ({placeholders});"
    cursor.execute(insert_statement, values)  # executes the INSERT with actual values
    inserted += 1


## commit and close connection
conn.commit()  
## .commit() is like hitting "save" - without this your inserts won't actually save

print(f"Done! {inserted} rows successfully inserted.")